<a href="https://colab.research.google.com/github/khushisidhu102-maker/aiml-30-days/blob/main/day01-02/demo_sms_spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ishansoni_sms_spam_collection_dataset_path = kagglehub.notebook_output_download('ishansoni/sms-spam-collection-dataset')

print('Data source import complete.')


# Spam Filter: Rules vs Machine Learning
**AI/ML Engineer · Day 1–2 · Live demo or student lab · 15–20 minutes**

In class, your team wrote if-else rules to catch spam SMS. Now we test those rules on **over 5,000 real messages**, then let a machine learning model learn its own rules from the same data. Which one wins?

**Works in both Kaggle and Google Colab.** Run cells top to bottom with **Shift + Enter**.
- **Kaggle:** click **+ Add Input**, search *SMS Spam Collection Dataset* (by UCI Machine Learning) and add it. The loader finds the file.
- **Colab:** nothing to add. The loader downloads a public copy of the same data.

## Step 1 · Load the messages

In [ ]:
import glob
import pandas as pd

kaggle_files = glob.glob("/kaggle/input/**/spam.csv", recursive=True)

if kaggle_files:
    # Kaggle: the attached dataset. Its file needs latin-1 encoding and has 3 empty extra columns.
    raw = pd.read_csv(kaggle_files[0], encoding="latin-1")
    df = raw.iloc[:, :2]
    df.columns = ["label", "text"]
    print("Loaded from Kaggle:", kaggle_files[0])
else:
    # Colab or laptop: a public copy of the same UCI SMS Spam Collection
    url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
    df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
    print("Loaded from the public copy on GitHub")

df = df.drop_duplicates(subset="text").reset_index(drop=True)
print("Messages (duplicates removed):", len(df))
df.sample(5, random_state=1)

## Step 2 · How much spam is there?
Before building anything, look at the balance. This number decides which metric we can trust.

In [ ]:
counts = df["label"].value_counts()
print(counts)
print(f"\nSpam share: {counts['spam'] / len(df):.1%}")
print(f"A 'model' that always says HAM would already be {counts['ham'] / len(df):.1%} accurate, and catch zero spam.")

## Step 3 · Your team's rules, tested on real data
Edit `SPAM_WORDS` to match the rules your team wrote in class, then run the cell. A message counts as spam if it contains **any** of the words.

In [ ]:
SPAM_WORDS = ["free", "win", "prize", "claim", "urgent", "cash", "http", "www"]   # <- edit me

def rule_says_spam(text):
    t = text.lower()
    return any(word in t for word in SPAM_WORDS)

df["rule_pred"] = df["text"].apply(lambda t: "spam" if rule_says_spam(t) else "ham")

is_spam = df["label"] == "spam"
flagged = df["rule_pred"] == "spam"
print(f"Spam caught : {(is_spam & flagged).sum()} of {is_spam.sum()}")
print(f"Spam missed : {(is_spam & ~flagged).sum()}")
print(f"False alarms: {(~is_spam & flagged).sum()} genuine messages wrongly flagged")

### Where do the rules break?

In [ ]:
pd.set_option("display.max_colwidth", 120)
print("GENUINE messages your rules flagged as spam:")
display(df[~is_spam & flagged][["text"]].head(5))
print("SPAM your rules missed:")
display(df[is_spam & ~flagged][["text"]].head(5))

**Discuss:** could you fix these by adding more words? What happens to the false alarms each time you add one?

## Step 4 · Let the machine learn the rules
Two pieces, glued into one pipeline:
- **CountVectorizer** turns each message into word counts (a *bag of words*): one column per word.
- **Multinomial Naive Bayes** learns how likely each word is in spam vs genuine messages.

Same 5-step pattern as Labs 1 and 2: load, split, fit, predict, evaluate.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

model = make_pipeline(CountVectorizer(lowercase=True), MultinomialNB())
model.fit(X_train, y_train)
print("Trained on", len(X_train), "messages. Testing on", len(X_test), "it has never seen.")

## Step 5 · Rules vs ML on the same test messages

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

def report(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=["spam", "ham"])
    caught, missed, false_alarms = cm[0, 0], cm[0, 1], cm[1, 0]
    total_spam = caught + missed
    print(f"{name:<22} accuracy {accuracy_score(y_true, y_pred):6.1%} | "
          f"spam caught {caught}/{total_spam} | false alarms {false_alarms}")

rule_test_pred = ["spam" if rule_says_spam(t) else "ham" for t in X_test]
report("Always say ham", y_test, ["ham"] * len(y_test))
report("Your rules", y_test, rule_test_pred)
report("Naive Bayes (ML)", y_test, model.predict(X_test))

## Step 6 · Look inside: which words did the model learn?
No one told the model these words. It found them by counting.

In [ ]:
import numpy as np

vec = model.named_steps["countvectorizer"]
nb = model.named_steps["multinomialnb"]
words = np.array(vec.get_feature_names_out())
spam_idx = list(nb.classes_).index("spam")
ham_idx = list(nb.classes_).index("ham")
spamminess = nb.feature_log_prob_[spam_idx] - nb.feature_log_prob_[ham_idx]

print("Top 15 words that push a message towards SPAM:")
print(", ".join(words[np.argsort(spamminess)[-15:][::-1]]))

## Step 7 · Try the messages from the slide, and your own
Add your own messages to the list. Try to fool the model.

In [ ]:
messages = [
    "Bhai, free ho toh call kar, notes chahiye for tomorrow's exam!",
    "Dear customer, your KYC is pending. Update now at kyc-update-bank.in or your account will be blocked today.",
    "Congrats! U hav w0n a 1Ph0ne. Cl!ck bit.ly/xyz",
    "Are we meeting at the canteen after the lab?",
    # add your own here
]

spam_col = list(model.classes_).index("spam")
for msg in messages:
    p_spam = model.predict_proba([msg])[0][spam_col]
    verdict = "SPAM" if p_spam >= 0.5 else "ham "
    rule = "spam" if rule_says_spam(msg) else "ham"
    print(f"ML: {verdict} ({p_spam:5.1%} spam) | rules: {rule:<4} | {msg[:70]}")

**Look closely.** Which message fooled both the rules *and* the model? Misspellings like `w0n` and `1Ph0ne` become words the model never saw in training, so it has no evidence either way.

This model also learned from UK messages from the 2000s, not Indian ones. Real spam filters look at character patterns as well as words, and are **retrained** on fresh, local data as scammers change tactics.

## Reflect (answer in a text cell or out loud)
1. Your rules and the model both had a test score. Which number would you show your manager, and which would you hide?
2. For a spam filter, which mistake is worse: missing a spam, or hiding a real message? Is that the same answer as the cancer screening lab?
3. Name one thing you would add to the training data before launching this in India.

## Save to GitHub (Colab)
**File → Save a copy in GitHub** → repo `aiml-30-days` → path `day01-02/demo_sms_spam.ipynb`.
On Kaggle: **File → Download notebook**, then upload it to the same GitHub folder.